# RAG Pipeline Demo — CS 763 Final Project

**Indirect Prompt Injection in Retrieval-Augmented Generation Systems**

This notebook demonstrates the baseline RAG pipeline:
1. Check Ollama is running
2. Load configuration
3. Load and inspect the NQ corpus
4. Initialize the pipeline (embed + index)
5. Run a sample query
6. Display retrieved chunks with scores
7. Display the LLM answer
8. Inspect the retrieval log

> **Environment:** Run inside the `rag-security` conda env (Python 3.11).

In [ ]:
# Cell 1: Check Ollama is running with llama3.2:3b
from rag.generator import Generator

gen = Generator(model="llama3.2:3b", host="http://localhost:11434")
reachable = gen.is_reachable()
print(f"Ollama reachable: {reachable}")
if not reachable:
    raise RuntimeError("Ollama is not running. Start it: ollama serve")
gen.assert_model_available()
print("llama3.2:3b is available.")

In [ ]:
# Cell 2: Load configuration
from rag.config import load_config

config = load_config("config.toml")
print(f"Seed: {config.seed}")
print(f"Top-k: {config.top_k}")
print(f"Corpus size: {config.corpus_size}")
print(f"Chunk words: {config.chunk_words}, Overlap: {config.overlap_words}")
print(f"Embed model: {config.embed_model}")
print(f"LLM model: {config.llm_model}")
print(f"ChromaDB path: {config.chroma_path}")
print(f"Collection: {config.collection}")

In [ ]:
# Cell 3: Load corpus and show stats
from rag.corpus import load_nq_passages, chunk_text

passages = load_nq_passages(n=config.corpus_size, seed=config.seed)
print(f"Loaded {len(passages)} unique passages")
print(f"
Sample passage (ID {passages[0].passage_id}):")
print(f"  Query: {passages[0].query}")
print(f"  Text:  {passages[0].text[:200]}...")

# Show chunking stats
all_chunks = []
for p in passages:
    all_chunks.extend(chunk_text(p.text, config.chunk_words, config.overlap_words))
print(f"
Total chunks: {len(all_chunks)}")
avg_words = sum(len(c.split()) for c in all_chunks) / len(all_chunks)
print(f"Avg words/chunk: {avg_words:.1f}")

In [ ]:
# Cell 4: Initialize pipeline (embed + index -- may take ~30s on first run)
from rag.pipeline import RAGPipeline

pipeline = RAGPipeline(config)
pipeline.build()  # loads corpus, indexes into ChromaDB, wires generator + logger
print(f"Pipeline initialized. Collection has {pipeline._retriever.count} chunks.")

In [ ]:
# Cell 5: Run a sample query
question = "what is the boiling point of water"
result = pipeline.query(question)
print(f"Query: {question}")
print(f"Number of hits: {len(result["hits"])}")

In [ ]:
# Cell 6: Display retrieved chunks with similarity scores
print(f"Retrieved chunks for: "{question}"
")
for hit in result["hits"]:
    print(f"--- Rank {hit["rank"]} | {hit["chunk_id"]} | Score: {hit["score"]:.4f} ---")
    print(hit["text"][:300])
    print()

In [ ]:
# Cell 7: Display LLM answer
print("=" * 60)
print("LLM ANSWER")
print("=" * 60)
print(result["answer"])

In [ ]:
# Cell 8: Inspect the retrieval log
import json
from pathlib import Path

log_path = Path(config.log_path)
if log_path.exists():
    lines = log_path.read_text(encoding="utf-8").strip().splitlines()
    print(f"Total log entries: {len(lines)}")
    last = json.loads(lines[-1])
    print(f"
Last entry:")
    print(f"  Timestamp: {last["timestamp"]}")
    print(f"  Query: {last["query"]}")
    print(f"  Top-k: {last["top_k"]}")
    print(f"  Results: {len(last["results"])} chunks")
else:
    print(f"Log file not found at {log_path}")

In [ ]:
# Cell 9: Cleanup
pipeline.close()
print("Pipeline closed.")